# 05. University of Arizona 4 km Daily SWE Snapshots (Auxiliary Diagnostic Archive)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/05_download_ua_swe_daily.ipynb)

This companion notebook downloads daily midnight-PST 4 km University of Arizona
SWE and snow depth snapshots cropped to the extended California snow-fed domain:
- **Output path**: `${FLOWRACLE_DATA_DIR}/ua-swe-daily-ca-region/ua_swe_daily_CA_all_months.zarr`

> **Note**: Flowracle's training and inference pipeline ingests
> the operational partial-monthly UA SWE dataset generated by
> `04_download_ua_swe_monthly.ipynb` (together with daily SNOTEL and CDEC snow
> pillow telemetry). This daily 4 km gridded archive is provided as an optional
> auxiliary dataset for high-frequency spatial snowpack diagnostics.

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0" "netCDF4>=1.6.0"

In [ ]:
import datetime
import gc
import os
import tempfile

import netCDF4  # noqa: F401
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "ua-swe-daily-ca-region")
os.makedirs(OUTPUT_DIR, exist_ok=True)
ZARR_STORE_PATH = os.path.join(OUTPUT_DIR, "ua_swe_daily_CA_all_months.zarr")

BASE_HTTP_URL = "https://climate.arizona.edu/data/UA_SWE/DailyData_4km"
LAT_MIN, LAT_MAX = 32.5, 45.5
LON_MIN, LON_MAX = -124.0, -115.0
START_DATE = "2011-11-01"
END_DATE = datetime.datetime.now().strftime("%Y-%m-%d")
VALID_MONTHS = set(range(1, 13))
BATCH_SIZE = 30

In [ ]:
def download_daily_file(year, month, day, temp_dir):
  wy = year + 1 if month >= 10 else year
  date_str = f"{year}{month:02d}{day:02d}"
  local_path = os.path.join(temp_dir, f"UA_SWE_{date_str}.nc")
  for suffix in ("stable", "provisional", "early"):
    url = f"{BASE_HTTP_URL}/WY{wy}/UA_SWE_Depth_4km_v1_{date_str}_{suffix}.nc"
    try:
      resp = requests.get(url, stream=True, timeout=60)
      if resp.status_code == 200:
        with open(local_path, "wb") as f:
          for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
        return local_path
    except requests.RequestException:
      continue
  return None


def load_and_crop_daily(local_file, date):
  with xr.open_dataset(local_file) as raw_ds:
    ds = raw_ds.load()
  numeric_vars = [
      v for v in ds.data_vars if np.issubdtype(ds[v].dtype, np.number)
  ]
  ds = ds[numeric_vars]
  ds["SWE"] = ds["SWE"] / 1000.0
  ds["SWE"].attrs["units"] = "m H2O"
  ds["DEPTH"] = ds["DEPTH"] / 1000.0
  ds["DEPTH"].attrs["units"] = "m"
  ds = (
      ds.squeeze("time", drop=True)
      .sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
      .rename({"lon": "longitude", "lat": "latitude"})
      .expand_dims(time=[pd.Timestamp(date)])
  )
  return ds


def save_batch_to_zarr(ds_list, zarr_path):
  if not ds_list:
    return
  combined = xr.concat(ds_list, dim="time")
  if not os.path.exists(zarr_path):
    enc = {
        v: {"chunks": (30,) + combined[v].shape[1:]} for v in combined.data_vars
    }
    combined.to_zarr(zarr_path, mode="w", encoding=enc)
  else:
    combined.to_zarr(zarr_path, mode="a", append_dim="time")

In [ ]:
all_dates = [
    d
    for d in pd.date_range(START_DATE, END_DATE, freq="D")
    if d.month in VALID_MONTHS
]
existing_times = set()
if os.path.exists(ZARR_STORE_PATH):
  ds_ex = xr.open_zarr(ZARR_STORE_PATH)
  existing_times = set(pd.to_datetime(ds_ex.time.values))
  ds_ex.close()

dates_todo = [d for d in all_dates if d not in existing_times]
print(f"Dates remaining to download: {len(dates_todo):,}")

batch = []
with tempfile.TemporaryDirectory() as temp_dir:
  for date in tqdm(dates_todo, desc="Downloading daily UA SWE"):
    nc_file = download_daily_file(date.year, date.month, date.day, temp_dir)
    if nc_file:
      try:
        batch.append(load_and_crop_daily(nc_file, date))
      finally:
        if os.path.exists(nc_file):
          os.remove(nc_file)
    if len(batch) >= BATCH_SIZE:
      save_batch_to_zarr(batch, ZARR_STORE_PATH)
      batch = []
      gc.collect()
  if batch:
    save_batch_to_zarr(batch, ZARR_STORE_PATH)

if os.path.exists(ZARR_STORE_PATH):
  print(xr.open_zarr(ZARR_STORE_PATH))